# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

# DONE: Load environment variables from .env file

In [4]:
# TODO: Load environment variables
load_dotenv()

True

### VectorDB Instance

In [5]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

In [7]:
# Create the collection if needed and reuse it on later notebook runs.
collection = chroma_client.get_or_create_collection(
    name="agentic_ai_udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [8]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    # Stable IDs plus upsert make this cell safe to run repeatedly.
    collection.upsert(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

print(f"Indexed {collection.count()} game documents in the persistent collection.")

Indexed 25 game documents in the persistent collection.


### Semantic search demonstration

Query the persistent collection with natural-language questions and inspect the ranked documents, metadata, and vector distances.

In [9]:
semantic_queries = [
    "When were Pokemon Gold and Silver released?",
    "What was the first three-dimensional Mario platform game?",
    "Which game is a FromSoftware open-world action RPG?",
]

for query in semantic_queries:
    results = collection.query(
        query_texts=[query],
        n_results=min(3, collection.count()),
        include=["documents", "metadatas", "distances"],
    )

    print(f"\nQuery: {query}")
    for rank, (document, metadata, distance) in enumerate(
        zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        ),
        start=1,
    ):
        print(
            f"  {rank}. {metadata.get('Name')} "
            f"({metadata.get('YearOfRelease')}, {metadata.get('Platform')}) "
            f"distance={distance:.4f}"
        )
        print(f"     {document}")


Query: When were Pokemon Gold and Silver released?
  1. Pokémon Gold and Silver (1999, Game Boy Color) distance=0.1134
     [Game Boy Color] Pokémon Gold and Silver (1999) - Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.
  2. Pokémon Ruby and Sapphire (2002, Game Boy Advance) distance=0.1475
     [Game Boy Advance] Pokémon Ruby and Sapphire (2002) - Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.
  3. Super Mario 64 (1996, Nintendo 64) distance=0.2232
     [Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.



Query: What was the first three-dimensional Mario platform game?
  1. Super Mario 64 (1996, Nintendo 64) distance=0.1147
     [Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.
  2. Super Mario World (1990, Super Nintendo Entertainment System (SNES)) distance=0.1447
     [Super Nintendo Entertainment System (SNES)] Super Mario World (1990) - A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.
  3. Mario Kart 8 Deluxe (2017, Nintendo Switch) distance=0.2015
     [Nintendo Switch] Mario Kart 8 Deluxe (2017) - An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.



Query: Which game is a FromSoftware open-world action RPG?
  1. Elden Ring (2022, PlayStation 5, Xbox Series X|S, PC) distance=0.1413
     [PlayStation 5, Xbox Series X|S, PC] Elden Ring (2022) - A challenging open-world action RPG developed by FromSoftware, set in the Lands Between and featuring exploration, character builds, and large-scale boss battles.
  2. Marvel's Spider-Man (2018, PlayStation 4) distance=0.1845
     [PlayStation 4] Marvel's Spider-Man (2018) - An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.
  3. The Legend of Zelda: Breath of the Wild (2017, Nintendo Switch) distance=0.1948
     [Nintendo Switch] The Legend of Zelda: Breath of the Wild (2017) - An open-world adventure that lets players explore Hyrule freely while solving physics-based puzzles and preparing for the battle against Calamity Ganon.
